# 02. 변수 생성 (마스터 데이터셋 생성)**통합 담당자만 관리하는 노트북.**각 팀원이 `src/features.py`에 추가한 함수들을 순서대로 호출하여모든 변수가 포함된 마스터 데이터셋(`apt_master.csv`)을 생성한다.**작업 순서:**1. 외부 데이터 7종을 `data/raw/`에 위치시킴 (Google Drive에서 다운로드)2. 이 노트북 전체 실행3. 결과 `data/processed/apt_master.csv`를 Google Drive에 공유**주의:** 카카오 API 호출이 포함된 함수가 있다면 시간이 오래 걸릴 수 있음 (단지 7,000개 × 4 시설 = 약 30분).

## 1. 라이브러리 로드

In [ ]:
import pandas as pdimport numpy as npimport sysimport osimport platformimport matplotlibif platform.system() == 'Darwin':    matplotlib.rcParams['font.family'] = 'AppleGothic'elif platform.system() == 'Windows':    matplotlib.rcParams['font.family'] = 'Malgun Gothic'else:    matplotlib.rcParams['font.family'] = 'NanumGothic'matplotlib.rcParams['axes.unicode_minus'] = Falsesys.path.append('../src')from features import (    add_transport_features,    add_education_features,    add_convenience_features,    add_negative_features,)

## 2. 전처리 데이터 로드

In [ ]:
df = pd.read_csv('../data/processed/apt_preprocessed.csv', encoding='utf-8-sig')print(f'전처리 데이터 shape: {df.shape}')print(f'컬럼: {df.columns.tolist()}')

## 3. 외부 데이터 로드각 팀원이 수집한 외부 데이터를 한 곳에서 로드.컬럼명/인코딩은 데이터에 따라 다르므로 확인 후 통일.

In [ ]:
# 교통 (담당: A)subway = pd.read_csv('../data/raw/지하철역.csv', encoding='cp949')bus    = pd.read_csv('../data/raw/버스정류장.csv', encoding='cp949')# 교육 (담당: B)school  = pd.read_csv('../data/raw/학교.csv', encoding='cp949')academy = pd.read_csv('../data/raw/학원.csv', encoding='cp949')# 생활편의 (담당: C)mart     = pd.read_csv('../data/raw/대형마트.csv', encoding='cp949')hospital = pd.read_csv('../data/raw/종합병원.csv', encoding='cp949')# 환경 (담당: D)funeral     = pd.read_csv('../data/raw/장례식장.csv', encoding='cp949')incinerator = pd.read_csv('../data/raw/소각장.csv', encoding='cp949')# 각 데이터 shape 확인for name, d in [('subway', subway), ('bus', bus), ('school', school),                ('academy', academy), ('mart', mart), ('hospital', hospital),                ('funeral', funeral), ('incinerator', incinerator)]:    print(f'{name:>12}: {d.shape}')

## 4. 외부 데이터 정제각 외부 데이터의 위도/경도 컬럼명을 통일 (`위도`, `경도`로).실제 컬럼명은 데이터에 따라 다르므로 필요시 rename.

In [ ]:
# 컬럼명 통일 예시 (실제 데이터 보고 수정)# subway = subway.rename(columns={'X': '경도', 'Y': '위도'})# bus = bus.rename(columns={'정류장위도': '위도', '정류장경도': '경도'})# 결측 좌표 제거for d_name, d in [('subway', subway), ('bus', bus), ('school', school),                  ('academy', academy), ('mart', mart), ('hospital', hospital),                  ('funeral', funeral), ('incinerator', incinerator)]:    before = len(d)    d.dropna(subset=['위도', '경도'], inplace=True)    print(f'{d_name}: {before} → {len(d)}건')

## 5. 변수 생성 (각 팀원 함수 호출)

In [ ]:
# 교통 변수 추가df = add_transport_features(df, subway_df=subway, bus_df=bus)# 교육 변수 추가df = add_education_features(df, school_df=school, academy_df=academy)# 생활편의 변수 추가df = add_convenience_features(df, mart_df=mart, hospital_df=hospital)# 환경(부정) 변수 추가df = add_negative_features(df, funeral_df=funeral, incinerator_df=incinerator)print(f'\n변수 추가 완료. 최종 shape: {df.shape}')print(f'컬럼: {df.columns.tolist()}')

## 6. 결측치 확인 및 처리

In [ ]:
# 변수별 결측치 확인missing = df.isnull().sum()missing = missing[missing > 0].sort_values(ascending=False)if len(missing) > 0:    print('결측치 발생 컬럼:')    print(missing)else:    print('결측치 없음')

In [ ]:
# 좌표 변환 실패 등으로 변수 결측인 행 제거 (선택)# 결측 비율이 낮으면 제거, 높으면 0/평균 대체 고려before = len(df)df = df.dropna()print(f'결측 행 제거: {before} → {len(df)}건')

## 7. 마스터 데이터셋 저장

In [ ]:
os.makedirs('../data/processed', exist_ok=True)df.to_csv('../data/processed/apt_master.csv', index=False, encoding='utf-8-sig')print('저장 완료: ../data/processed/apt_master.csv')print(f'최종 shape: {df.shape}')print(f'\n이 파일을 Google Drive에 업로드하여 팀원들과 공유.')

## 8. 마스터 데이터셋 요약다음 노트북(03_experiments.ipynb)에서 사용할 변수 목록 정리.

In [ ]:
# 변수 그룹별 컬럼 정리 (03 노트북 슬롯 구성용)PHYSICAL    = ['전용면적(㎡)', '층', '노후도']REGION      = ['강남권']TRANSPORT   = [c for c in df.columns if c.startswith(('subway_', 'bus_'))]EDUCATION   = [c for c in df.columns if c.startswith(('school_', 'academy_'))]CONVENIENCE = [c for c in df.columns if c.startswith(('mart_', 'hospital_'))]NEGATIVE    = [c for c in df.columns if c.startswith(('funeral_', 'incinerator_'))]print('=== 변수 그룹 요약 ===')print(f'PHYSICAL    ({len(PHYSICAL)}): {PHYSICAL}')print(f'REGION      ({len(REGION)}): {REGION}')print(f'TRANSPORT   ({len(TRANSPORT)}): {TRANSPORT}')print(f'EDUCATION   ({len(EDUCATION)}): {EDUCATION}')print(f'CONVENIENCE ({len(CONVENIENCE)}): {CONVENIENCE}')print(f'NEGATIVE    ({len(NEGATIVE)}): {NEGATIVE}')